In [4]:
import numpy as np
import pandas as pd
from wavediffusion.waveana import add_lat_lon 
import xarray as xr

### Assemble data into netcdf

In [8]:
# Sample scenario to analyze
year = 2004
month = 4
path = f'/global/homes/j/jiarongw/scratch_folder/final/temp/OPTION1_moredata_sigma100_epoch20_{year}{month:02d}/'

for i, index in enumerate(range(0, 20, 8)):
    x_truth = np.load(f'{path}truth_{index}.npy')
    x_mean = np.load(f'{path}mean_{index}.npy')
    x_sample = np.load(f'{path}sample_{index}.npy')
    f = np.load(f'{path}forcing_{index}.npy') 
    icymask = np.load(f'{path}icymask_{index}.npy')

    # Time 
    day = int(index / 8 + 6)
    date = pd.Timestamp(year=year, month=month, day=1) + pd.Timedelta(days=day - 1)
    
    wsp = (f[0]**2+f[1]**2)**0.5
    
    if i == 0:
        ds = add_lat_lon (np.concatenate((x_truth, x_sample, wsp[np.newaxis, :])), 
                          ('hs_truth', 'tm_truth', 'dir_truth', 'hs_sample', 'tm_sample', 'dir_sample', 'wsp'))
        ds = ds.expand_dims(time=[date])
    else:
        ds_ = add_lat_lon (np.concatenate((x_truth, x_sample, wsp[np.newaxis, :])), 
                          ('hs_truth', 'tm_truth', 'dir_truth', 'hs_sample', 'tm_sample', 'dir_sample', 'wsp'))
        ds_ = ds_.expand_dims(time=[date])
        ds = xr.concat([ds, ds_], dim="time")

In [9]:
# Notes for each variable
var_notes = {
    'hs_truth':  {'long_name': 'Significant wave height (truth)', 'units': 'm'},
    'tm_truth':  {'long_name': 'Mean wave period (truth)', 'units': 's'},
    'dir_truth': {'long_name': 'Mean wave direction (ERA5/truth)', 'units': 'deg'},
    'hs_sample': {'long_name': 'Significant wave height (diffusion sample)', 'units': 'm'},
    'tm_sample': {'long_name': 'Mean wave period (diffusion sample)', 'units': 's'},
    'dir_sample':{'long_name': 'Mean wave direction (diffusion sample)', 'units': 'deg'},
    'wsp':       {'long_name': 'Wind speed magnitude', 'units': 'm/s'},
}

for var, attrs in var_notes.items():
    ds[var].attrs.update(attrs)

In [10]:
ds.to_netcdf(f'{path}sample_data_{year}{month:02d}.nc')

### Check OSN pods

In [3]:
import boto3

session = boto3.Session(profile_name="osn")
s3 = session.client("s3", endpoint_url="https://nyu1.osn.mghpcc.org/")

paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="m2lines-pubs", Delimiter="/"):
    for prefix in page.get("CommonPrefixes", []):
        print("DIR:", prefix["Prefix"])
    for obj in page.get("Contents", []):
        print("FILE:", obj["Key"], obj["Size"])

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


DIR: FOMO/
DIR: GFDL-CM2.6/
DIR: Google-CM2.6/
DIR: Samudra/


### Upload data

In [14]:
year = 2004
month = 9
path = f'/global/homes/j/jiarongw/scratch_folder/final/temp/OPTION1_moredata_sigma100_epoch20_{year}{month:02d}/'

session = boto3.Session(profile_name="osn")
s3 = session.client("s3", endpoint_url="https://nyu1.osn.mghpcc.org/")

s3.upload_file(
    f"{path}sample_data_{year}{month:02d}.nc",
    "m2lines-pubs",
    f"Wave/sample_data_{year}{month:02d}.nc"   # this "Wave/" prefix is what makes it show up as a folder
)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


In [15]:
response = s3.list_objects_v2(Bucket="m2lines-pubs", Prefix="Wave/")
for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"], obj["LastModified"])

Wave/sample_data_200404.nc 19371180 2026-07-07 22:53:28.966000+00:00
Wave/sample_data_200409.nc 19371180 2026-07-07 22:54:30.940000+00:00
